# Ghosting EDA (analytics/ghosting)

Tables only: this environment has no plotting library. Every number from the fixture run below is **fixture output, not a result**; point `GAMES` at `validate.iter_data_root(DATA_ROOT)` on the Mac for the real tables. The functions live in `analytics/ghosting/eda.py`; the trainer context comes from `analytics.ghosting.train.run`, which runs the held-out guard and `schema.assert_time_forward` before any fit.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'analytics' / 'ghosting').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from analytics.ghosting import eda, features as F, train as T, validate as V
pd.set_option('display.width', 160)


## Inputs

Fixture games (small, so the notebook runs in seconds). Replace with `V.iter_data_root('<Drive>/nba_harvest')` for the real run.


In [ ]:
GAMES = V.fixture_games(seed=0, n_games=4, n_possessions=6, heldout_every=4)
for g in GAMES:
    g['validation'] = V.validate_game(g)
pd.DataFrame([{'game': g['sportvu_game'], 'heldout': g['heldout'], **{'poss_' + k: v for k, v in g['validation']['possessions'].items() if k != 'schema_problems'}} for g in GAMES])


## The run

Out-of-fold under grouped folds by game; the held-out game is set aside before any fit and scored once at the end.


In [ ]:
CTX = T.run(GAMES, n_folds=5, seed=0, n_boot=100)
print('fixture output, not a result')
print('train games', CTX['train_games'], 'held out', CTX['heldout_games'], 'rows', CTX['train_rows'])
print('decision', CTX['decision'])


## Displacement distributions

The 0.2 s displacement target, the defender speed and the distance to the assignment at t minus 1.


In [ ]:
eda.displacement_distributions(CTX['frame'][~CTX['frame']['heldout']])


In [ ]:
eda.feature_summary(CTX['frame'][~CTX['frame']['heldout']])


## Error by horizon

Out-of-fold mean, median and p90 position error per candidate; the report carries the bootstrap intervals and the paired tests.


In [ ]:
eda.error_by_horizon(CTX['errors'])


In [ ]:
pd.DataFrame([{'pair': p, 'horizon_s': h, **r} for p, byh in CTX['pairs'].items() for h, r in byh.items()])


## Deviation by end kind

Mean per-possession deviation (actual minus ghost distance to the assignment, 1.0 s rows of the selected ghost) by the possession's end kind. `end_kind` and `points` are section 19 targets, joined here after every fit.


In [ ]:
POSS = pd.concat([g['possessions'] for g in GAMES], ignore_index=True)
eda.deviation_by_end_kind(CTX['ghost_possessions'], POSS)


## Correlation table

Per-game Spearman of mean deviation against points, and the sign test over games.


In [ ]:
per_game, sign = eda.correlation_table(CTX['ghost_possessions'])
print(sign)
print(eda.markdown(per_game))
per_game


## Held-out game (scored once)


In [ ]:
pd.DataFrame(CTX['heldout_scores']) if CTX['heldout_scores'] else 'no held-out game in this run'
